In [2]:
!pip install -qU langchain langchain_openai langchain-core langchain-community langgraph psycopg[binary,pool]==3.2.6

In [3]:
import os
with open("/content/api_key_pic24_openai.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

with open("/content/postgrest.txt") as archivo:
  uribd = archivo.read()

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from langchain_core. prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import requests

In [5]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

## Usar una herramienta SQL Predefinida

In [7]:
#Creamos la conexion a la base de datos
db_data = SQLDatabase.from_uri(uribd)

# Herramienta BD
toolkit_bd = SQLDatabaseToolkit(db=db_data,llm=ChatOpenAI(temperature=0))
tools_bd = toolkit_bd.get_tools()

## Creacion de una herramienta SQL personalizada

## Herramienta personalizada SQL

In [8]:
def seleccionar_tabla_v2(question: str) -> str:
    """
    Selecciona la tabla correcta basada en la pregunta utilizando la LLM para inferir el nombre de la tabla,
    y luego verifica si el nombre de la tabla existe en las tablas disponibles.
    """
    # Obtiene las tablas disponibles desde la base de datos
    tablas_disponibles = db_data.get_table_names()  # Listado dinámico de tablas

    # Crea un mensaje que pasa la lista de tablas disponibles al modelo de lenguaje
    tablas_str = ", ".join(tablas_disponibles)  # Las tablas disponibles como un string

    # Consultamos a la LLM para obtener el nombre de la tabla basado en la pregunta y las tablas disponibles
    prompt = (f"Las siguientes tablas están disponibles en la base de datos: {tablas_str}. "
              f"Segun las tablas indicadas  ¿Devuelve sólo el nombre de la tabla a  la que corresponde esa pregunta: '{question}'?")
    model = ChatOpenAI(verbose=True)
    # Suponiendo que 'model' es el objeto que invoca el modelo de lenguaje
    #nombre_tabla_sugerido = model(prompt).strip().lower()  # Respuesta procesada a minúsculas
    respuesta = model.invoke([HumanMessage(content=prompt)])
    nombre_tabla_sugerido = respuesta.content.strip().lower()

    # Validamos si la tabla sugerida por el modelo existe en la base de datos
    #if nombre_tabla_sugerido not in tablas_disponibles:
    #    return f"[ERROR] La tabla '{nombre_tabla_sugerido}' no se reconoce. Las tablas disponibles son: {', '.join(tablas_disponibles)}"

    return nombre_tabla_sugerido

In [9]:
# Función para seleccionar la tabla correcta basada en la pregunta
def seleccionar_tabla(question: str) -> str:
    # Define las tablas disponibles
    tablas_disponibles = ['temperatura_registros', 'humedad_registros', 'calidad_aire_registros', 'sensores', 'historial_dispositivos']

    # Aquí puedes agregar lógica de selección, por ejemplo, buscar palabras clave en la pregunta
    if "temperatura" in question.lower():
        return "temperatura_registros"
    elif "humedad" in question.lower():
        return "humedad_registros"
    elif "calidad de aire" in question.lower():
        return "calidad_aire_registros"
    elif "sensor" in question.lower():
        return "sensores"
    elif "historial" in question.lower():
        return "historial_dispositivos"
    else:
        # Si no se encuentra una coincidencia clara, puede retornar una tabla predeterminada o lanzar un error
        return "temperatura_registros"  # Tabla predeterminada

In [ ]:
print(db_data.get_usable_table_names())

['calidad_aire_registros', 'historial_dispositivos', 'humedad_registros', 'productos', 'sensores', 'temperatura_registros']


In [10]:
from pydantic import BaseModel, Field
from langchain_core.tools import Tool


db_data = SQLDatabase.from_uri(uribd)
model = ChatOpenAI(verbose=True)
#model = ChatOpenAI(model="gpt-4o-mini", verbose=True)

#@tool(args_schema=GetSchemaInput)
@tool
def get_schema(question: str) -> str:
    "Herramienta para recuperar el esquema de una tabla específica,si la tabla no existe devuelve un mensaje de error controlado"
    # Usa la función 'seleccionar_tabla' para obtener el nombre de la tabla basándose en la pregunta
    tabla_seleccionada = seleccionar_tabla_v2(question)
    # Obtiene las tablas disponibles desde la base de datos
    tablas_disponibles = db_data.get_table_names()
    # Si la tabla seleccionada no existe en la base de datos, devuelve un mensaje de error controlado
    if tabla_seleccionada not in tablas_disponibles:
        return f"[ERROR] La tabla '{tabla_seleccionada}' no se reconoce. Las tablas disponibles son: {', '.join(tablas_disponibles)}"
    # Si la tabla es válida, devuelve el esquema de la tabla seleccionada
    schema = db_data.get_table_info([tabla_seleccionada])
    return schema

#def get_schema(table_name: str) -> str:
#    "Herramienta para recuperar el esquema de una tabla específica,si la tabla no existe devuelve un mensaje de error controlado"
#    tablas_disponibles = db_data.get_table_names()
#    if table_name not in tablas_disponibles:
#        return f"[ERROR] La tabla '{table_name}' no se reconoce. Las tablas disponibles son: {', '.join(tablas_disponibles)}"
#    return db_data.get_table_info([table_name])

#def get_schema(table_name: str) -> str:
#  "Herramienta para recuperar el esquema solo de la tabla indicada"
#  return db_data.get_table_info([table_name])



promptsql = ChatPromptTemplate.from_template("""
Basandonos en el esquema de tabla siguiente, escribe una consulta SQL que responda a la pregunta del usuario:
    Tabla: {table}

    Esquema: {schema}

    Pregunta: {question}
    Sql Query:
""")

sqlchain = (
    promptsql
    | model.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

@tool
def generar_sql(question: str) -> str:
    """Genera una consulta SQL a partir del esquema, la pregunta y  usando la tabla adecuada seleccionada dinámicamente """
    # Selecciona la tabla basándose en la pregunta
    tabla_seleccionada = seleccionar_tabla(question)
    schema = db_data.get_table_info([tabla_seleccionada])  # Solo obtiene el esquema de la tabla seleccionada
    return sqlchain.invoke({"schema": schema, "question": question,"table": tabla_seleccionada})



@tool
def run_query(query) -> str:
    """Herramienta que ejecuta una consulta SQL en la base de datos"""
    resultado = db_data.run(query)
    if not resultado or resultado == [(None,)]:
        return "[SIN RESULTADOS] No se encontraron registros que coincidan con la consulta."
    return resultado
#def run_query(query) -> str:
#  """Herramienta que ejecuta una consulta SQL en la base de datos"""
#  return db_data.run(query)


promptsqlquery = ChatPromptTemplate.from_template(
    """
    Basandonos en el esquema de tabla inferior, pregunta, SQL Query y Respuesta, escribe una respuesta en lenguaje natural:
    Tabla: {table}

    Esquema: {schema}

    Pregunta: {question}
    Sql Query: {sql_query}
    SQL Respuesta: {response}

    """)

sqlnatural_chain = (
     promptsqlquery
    | model
    | StrOutputParser()
)

@tool(return_direct=True)
def generar_respuesta(question: str, sql_query: str, response: str) -> str:
    """Genera una respuesta en lenguaje natural tomando el esquema, la consulta, la query,la respuesta de la ejecucion de la query y la tabla seleccionada"""
    tabla_seleccionada = seleccionar_tabla(question)
    schema = db_data.get_table_info([tabla_seleccionada])
    return sqlnatural_chain.invoke({"schema": schema, "question": question, "sql_query":sql_query, "response":response, "table": tabla_seleccionada  })


## Agente SQL

In [11]:
model = ChatOpenAI(verbose=True)
#model = ChatOpenAI(model="gpt-4o-mini", verbose=True)
memory = MemorySaver()
#construccion del prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """ Eres un asistente de base de datos, utiliza tus herramientas para resolver las preguntas de los usuarios
        """),

     ("human", "{messages}"),

     ]
)

#toolkit = [get_schema,generar_sql,run_query,generar_respuesta]
toolkit = [get_schema,generar_sql,run_query,generar_respuesta]
agent2 = create_react_agent(model, toolkit, checkpointer=memory, prompt=prompt)

In [14]:
# Use the agent
config = {"configurable": {"thread_id": "41900_sqltablacasa_4101019_567_85"}}
for step in agent2.stream(
    #{"messages": [HumanMessage(content="Qué temperatura registró la tabla temperatura_registros para el ambiente 'cocina' el 29 de abril del 2025?")]},
    #{"messages": [HumanMessage(content=". ¿Qué nivel de calidad de arire se ha registrado en la tabla calidad_aire_registros el 29 de abril del 2025?")]},
    #{"messages": [HumanMessage(content= "¿En qué ambientes está instalado el sensor de humedad según la tabla sensores?")]},
    #{"messages": [HumanMessage(content= "¿Qué dispositivos ha usado Juan esta mañana según los registros de la tabla historial_dispositivos?")]},
    #{"messages": [HumanMessage(content= "¿Qué temperatura promedio hubo en el dormitorio esta semana?")]},
    #{"messages": [HumanMessage(content= "¿Dónde se ha registrado la mayor humedad en los ultimos 3 dias")]},
    #{"messages": [HumanMessage(content= "¿Cuál fue la zona con peor calidad de aire de los últimos 2 dias?")]},
    {"messages": [HumanMessage(content= "¿Qué sensores hay instalados en la cocina?")]},
    #{"messages": [HumanMessage(content= "¿Indica todas las personas que  apagaron el ventilador en la cocina segun el historial?")]},
    #{"messages": [HumanMessage(content= "¿Indica todas las personas que  apagaron las luces en la cocina segun el historial?")]},

    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

¿Qué sensores hay instalados en la cocina?
================================== Ai Message ==================================
Tool Calls:
  get_schema (call_Hq5oJy7PsYfNQaKOaq3xRlqC)
 Call ID: call_Hq5oJy7PsYfNQaKOaq3xRlqC
  Args:
    question: sensores instalados en la cocina
================================= Tool Message =================================
Name: get_schema

[ERROR] La tabla 'la tabla correspondiente a la pregunta 'sensores instalados en la cocina' es la tabla de sensores.' no se reconoce. Las tablas disponibles son: calidad_aire_registros, historial_dispositivos, humedad_registros, productos, sensores, temperatura_registros
================================== Ai Message ==================================
Tool Calls:
  get_schema (call_gOnbI0e8g5cwUwTshgfoFNez)
 Call ID: call_gOnbI0e8g5cwUwTshgfoFNez
  Args:
    question: sensores instalados en la cocina
================================= Tool

In [15]:
step["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: generar_respuesta

En la cocina hay instalados un sensor de temperatura, un sensor de humedad y un sensor de calidad de aire.
